# Overview
Calling Gemini on Vertex AI: once through an ADK agent, once through the `google-genai` client.


In [ ]:
# @title Loads the shared env vars from the root directory
import os

# Allows running async functions
import nest_asyncio
from nb_settings import settings

nest_asyncio.apply()

# Use Vertex AI with Google Cloud credentials, not an API key
os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "true")
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", settings.region)

print(f"Using the model: {settings.llm_model_name}")

## ADK agent
The same `Agent` setup as `agent_service/agents/`, run in memory.


In [ ]:
from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

agent = Agent(
    name="poet",
    model=settings.llm_model_name,
    instruction="You write short ballads.",
)
runner = InMemoryRunner(agent=agent, app_name="notebook")
session = await runner.session_service.create_session(app_name="notebook", user_id="me")

message = types.Content(
    role="user", parts=[types.Part(text="Write me a ballad about the stars")]
)
async for event in runner.run_async(
    user_id="me", session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content:
        print(event.content.parts[0].text)

## google-genai client
A single model call, no agent.


In [ ]:
from google import genai

client = genai.Client()

response = await client.aio.models.generate_content(
    model=settings.llm_model_name, contents="Write a poem about the stars."
)
print(response.text)